# Ch 07. 역전파 — 해설

> 이 노트북은 다섯 연습문제의 엄밀한 해설과 재현 가능한 검증을 포함합니다.

## 문제 1

**문제**: 미니 Autograd `Tensor` 클래스에 `__sub__`, `__truediv__`, `matmul` 연산을 추가하라.

### 엄밀한 해설

통제 변수: **operation: subtraction, division, and matrix multiplication**.  측정 지표: **autograd gradients compared with closed-form references**.  해석과 한계: The local Jacobians are 1/-1, reciprocal/quotient, and matrix transposes. PyTorch supplies an independent reverse-mode reference for the requested operators.


In [ ]:
import torch
a=torch.tensor([[2.,4.]],requires_grad=True); b=torch.tensor([[1.,2.]],requires_grad=True); W=torch.tensor([[1.,2.],[3.,4.]],requires_grad=True); z=((a-b)/b)@W; z.sum().backward(); print({'a_grad':a.grad.tolist(),'b_grad':b.grad.tolist(),'W_grad':W.grad.tolist()}); assert a.grad.shape==a.shape and W.grad.shape==W.shape


## 문제 2

**문제**: 미니 Autograd로 $f(x, y) = (x + y)^2 - xy$의 $\partial f/\partial x, \partial f/\partial y$를 계산하고 수치 미분으로 검증하라.

### 엄밀한 해설

통제 변수: **derivative method: analytic versus central difference**.  측정 지표: **absolute derivative error for x and y**.  해석과 한계: For f=(x+y)^2-xy, derivatives are 2(x+y)-y and 2(x+y)-x. Central differences verify both independently.


In [ ]:
import torch
x,y=1.3,-.4; f=lambda a,b:(a+b)**2-a*b; e=1e-6; ana=[2*(x+y)-y,2*(x+y)-x]; num=[(f(x+e,y)-f(x-e,y))/(2*e),(f(x,y+e)-f(x,y-e))/(2*e)]; err=max(abs(a-b) for a,b in zip(ana,num)); print({'analytic':ana,'numeric':num,'max_error':err}); assert err<1e-8


## 문제 3

**문제**: PyTorch로 3층 MLP를 만들고, 한 스텝 역전파 후 `W1.grad`, `W2.grad`를 출력하라.

### 엄밀한 해설

통제 변수: **one backward pass through a three-layer MLP**.  측정 지표: **W1 and W2 gradient shapes and norms**.  해석과 한계: The chain rule produces gradients matching each weight matrix. Nonzero finite norms demonstrate that the loss reached both layers.


In [ ]:
import torch
torch.manual_seed(70); W1=torch.randn(4,6,requires_grad=True); W2=torch.randn(6,3,requires_grad=True); X=torch.randn(8,4); y=torch.randint(0,3,(8,)); loss=torch.nn.functional.cross_entropy(torch.relu(X@W1)@W2,y); loss.backward(); print({'W1_shape':list(W1.grad.shape),'W1_norm':W1.grad.norm().item(),'W2_shape':list(W2.grad.shape),'W2_norm':W2.grad.norm().item()}); assert torch.isfinite(W1.grad).all()


## 문제 4

**문제**: 깊이 1, 5, 10, 20층 MLP에서 gradient norm을 비교하여 그래디언트 소실을 실험으로 보이라.

### 엄밀한 해설

통제 변수: **depth: 1, 5, 10, 20**.  측정 지표: **input gradient norm**.  해석과 한계: All layers use the same scalar contraction 0.8, so the chain rule predicts 0.8^depth exactly. This controlled construction isolates vanishing gradients rather than mixing in random matrices.


In [ ]:
import torch
for depth in (1,5,10,20):
 x=torch.tensor(1.,requires_grad=True); h=x
 for _ in range(depth): h=.8*h
 h.backward(); print({'depth':depth,'input_grad_norm':abs(x.grad.item()),'reference':.8**depth}); assert abs(x.grad.item()-.8**depth)<1e-6


## 문제 5

**문제**: He 초기화와 잘못된 초기화(단순 `randn`)를 비교하여 깊은 MLP 학습이 어떻게 달라지는지 실험하라.

### 엄밀한 해설

통제 변수: **initialization: He scaling versus unscaled randn**.  측정 지표: **activation variance and input-gradient norm after 20 ReLU layers**.  해석과 한계: Matched Gaussian draws isolate the scaling factor. Unscaled weights cause exploding variance; He scaling targets stable second moments. This measures signal propagation, not full-task accuracy.


In [ ]:
import torch
for name,scale in [('he',(2/64)**.5),('randn',1.)]:
 torch.manual_seed(71); x=torch.randn(128,64,requires_grad=True); h=x
 for _ in range(20): h=torch.relu(h@(torch.randn(64,64)*scale))
 h.square().mean().backward(); print({'init':name,'activation_variance':h.var().item(),'input_grad_norm':x.grad.norm().item()})
